# Phase 3: Published Feature-View Reproduction

This notebook compares the independently reproduced static feature views. All text statistics and embeddings are fitted on the training partition only. Validation and final-test contracts are transformed with frozen training state. No model is trained here.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

summary_path = PROJECT_ROOT / "results" / "baseline" / "feature_view_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        "Run '.\\.venv\\Scripts\\python.exe -m src.features.reproduce' first."
    )

summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary["anti_leakage_policy"]

## Dimensions And Sparsity

In [ ]:
rows = []
for view_name, view in summary["feature_views"].items():
    for split_name, metrics in view["splits"].items():
        rows.append({
            "view": view_name,
            "split": split_name,
            "rows": metrics["rows"],
            "dimensions": metrics["dimensions"],
            "density": metrics.get("density"),
            "sparsity": metrics.get("sparsity"),
        })
feature_table = pd.DataFrame(rows)
feature_table

In [ ]:
train_dimensions = feature_table[feature_table["split"] == "train"].set_index("view")["dimensions"]
ax = train_dimensions.plot.bar(color="#4C78A8", logy=True)
ax.set_title("Feature dimensions fitted from training data")
ax.set_xlabel("Feature view")
ax.set_ylabel("Dimensions (log scale)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

In [ ]:
sparse_views = feature_table.dropna(subset=["sparsity"])
sparsity_pivot = sparse_views.pivot(index="view", columns="split", values="sparsity")
ax = sparsity_pivot[["train", "validation", "test"]].plot.bar()
ax.set_title("Sparsity by feature view and split")
ax.set_xlabel("Feature view")
ax.set_ylabel("Sparsity")
ax.set_ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

## Feature Distributions

In [ ]:
distribution_rows = []
for view_name, view in summary["feature_views"].items():
    for split_name, metrics in view["splits"].items():
        if "nonzero_per_contract" in metrics:
            distribution_rows.append(
                {"view": view_name, "split": split_name, **metrics["nonzero_per_contract"]}
            )
pd.DataFrame(distribution_rows)

In [ ]:
word2vec = summary["feature_views"]["opcode_word2vec_mean"]
pd.DataFrame([
    {"split": split_name, **metrics["row_l2_norm"], "zero_vectors": metrics["zero_vector_contracts"]}
    for split_name, metrics in word2vec["splits"].items()
])

In [ ]:
creator = summary["feature_views"]["chronological_creator"]
creator_table = pd.DataFrame.from_dict(creator["splits"], orient="index")
creator_table.index.name = "split"
creator_table.reset_index()

In [ ]:
ax = creator_table["known_prior_positive_creator_share"].plot.bar(color="#F58518")
ax.set_title("Contracts flagged by chronological creator feature")
ax.set_xlabel("Split")
ax.set_ylabel("Share")
plt.xticks(rotation=0)
plt.tight_layout()

The fitted objects are reusable preprocessing artifacts. Feature matrices are deliberately not persisted. MulCas, classifiers, and evaluation are outside Phase 3.